# 3. Evaluation

`layaft val` measures any checkpoint on the task's **hand-written test set**, never seen in training, with metrics per
question type:

- `choice`: accuracy and ECE (how far confidence is from accuracy; 0 is perfect), and the traffic light when the
  question has `review` thresholds (green is assigned alone, yellow a person confirms, red a person decides).
- `score`: exact and within one level. `noul`: accuracy at 0.5 and Brier.

With `ctx=` the same cases are also wrapped in held-out filler at 8k, 16k, 32k… up to the context, rotating the case
between the start, the middle and the end: if the long rows fall behind the short one, the context is not trained yet.

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")  # data/, tasks/ and runs/ live in the repo root.
from layaft import LayaFT, Task

task = Task.load("helpdesk")

from IPython.display import Markdown, SVG, display
from layaft.evaluate import report

results = {name: LayaFT(model).val(task, save=False)["summary"]
           for name, model in {"base": "multilingual", "tuned": "runs/helpdesk-1k"}.items()}
display(Markdown(report.table(task, results)))
report.chart(task, results, Path("runs/compare.svg"), subtitle="20 hand-written test tickets")
display(SVG(filename="runs/compare.svg"))

## By length

In [ ]:
out = LayaFT("runs/helpdesk-32k").val(task, ctx="32k")
display(Markdown(report.table(task, out["by_length"])))

## Try your own case

In [ ]:
answers = LayaFT("runs/helpdesk-1k").predict(
    {"ticket": "Desde esta mañana nadie del almacén puede imprimir etiquetas de envío: la impresora Zebra "
               "muestra «Error de cabezal» y ya la reiniciamos dos veces. Hoy salen 80 pedidos."}, task)
print("category:", answers["categoria"]["choice"], f"({100 * answers['categoria']['confidence']:.0f} %)")
print("priority:", round(answers["prioridad"]["score"], 2), "· blocking:", round(answers["bloqueo"]["noul"], 2))